In [ ]:
import sqlite3


# Setup the database
def setup_database(db_path: str = 'literature.db'):
    """
    Function to create the SQLite database, set up the connection,
    and create tables if they do not already exist.
    """
    connector = sqlite3.connect(db_path)
    cursor = connector.cursor()

    # Main table for papers
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS papers (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        doi TEXT UNIQUE NOT NULL,
        title TEXT,
        publication_year INTEGER,
        authors TEXT,
        venue TEXT,
        volume TEXT,
        publication_type TEXT,
        publication_source TEXT,
        processed BOOLEAN DEFAULT 0,
        file_path TEXT DEFAULT NULL
    )
    """)

    # Table for paper assessments
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS paper_assessments (
        paper_id INTEGER PRIMARY KEY,
        is_neurosymbolic BOOLEAN,
        is_development BOOLEAN,
        paper_type TEXT,
        summary TEXT,
        takeaways TEXT,
        assessment_date TIMESTAMP,
        FOREIGN KEY (paper_id) REFERENCES papers (id)
    )
    """)

    # Table for keywords
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS keywords (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        keyword TEXT UNIQUE
    )
    """)

    # Relationship table for keywords and papers
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS rel_keywords_papers (
        paper_id INTEGER,
        keyword_id INTEGER,
        FOREIGN KEY (paper_id) REFERENCES papers (id),
        FOREIGN KEY (keyword_id) REFERENCES keywords (id)
    )
    """)

    connector.commit()
    connector.close()

In [ ]:
from PyPDF2 import PdfReader


def extract_text_from_pdf(file_path: str) -> str:
    """
    Extract the full text from a PDF using PyPDF2.
    """
    try:
        reader = PdfReader(file_path)
        all_text = []
        for page in reader.pages:
            page_text = page.extract_text() or ""
            all_text.append(page_text)
        return "\n".join(all_text)
    except Exception as e:
        print(f"Error extracting text from {file_path}: {e}")
        return ""


def get_first_page_text(file_path: str) -> str:
    """
    Extracts text from the first page of a PDF.
    Returns an empty string if no pages exist or an error occurs.
    """
    try:
        reader = PdfReader(file_path)
        if len(reader.pages) > 0:
            first_page = reader.pages[0]
            return first_page.extract_text() or ""
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return ""


CLASSIFICATION_PROMPT = """
You are classifying a research paper on whether it focuses on 'development' (e.g.,
software development, systems development, or methodological development) or not.

Information we have:
Title: {title}
Abstract: {abstract}

If more information is needed, you can use the PaperRetriever tool.

Answer "YES" if the paper is about development, "NO" if not.
"""

In [ ]:
from langchain.vectorstores import Chroma
from langchain.agents import Tool


# Setup the agent and tools
def create_paper_retriever_tool(vectorstore: Chroma) -> Tool:
    """
    Create a tool that can be called by the agent to do a similarity search
    over the papers' text.
    """
    def retrieval_tool(query: str) -> str:
        docs = vectorstore.similarity_search(query, k=2)
        contents = "\n\n".join([d.page_content for d in docs])
        return contents

    return Tool(
        name="PaperRetriever",
        func=retrieval_tool,
        description="Retrieves relevant text from the stored papers for the query."
    )

In [ ]:
from dotenv import load_dotenv, find_dotenv
from datetime import datetime
from langchain.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.docstore.document import Document
from langchain.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, AgentType


load_dotenv(find_dotenv())
db_path: str = 'literature.db'

# Set up the DB (if not existing)
setup_database(db_path)

# Connect to DB
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Initialize OpenAI LLM and embeddings
llm = ChatOpenAI(
    model="gpt-4o-mini", # gpt-4-turbo gpt-4o-mini
    temperature=0.0,
    seed=3459746589468594
)
embeddings = OpenAIEmbeddings()

# Initialize or load Chroma store
persist_directory = "./chroma_store"
vectorstore = Chroma(
    collection_name="papers_collection",
    embedding_function=embeddings,
    persist_directory=persist_directory
)

# Create the retriever tool
retriever_tool = create_paper_retriever_tool(vectorstore)

# Build the agent
agent = initialize_agent(
    tools=[retriever_tool],
    llm=llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Get papers that are not processed yet
# (or you can do WHERE processed=0, or any custom condition)
cursor.execute("""
    SELECT id, title, file_path
    FROM papers
    WHERE file_path IS NOT NULL
    AND id = 7
""")
paper_rows = cursor.fetchall()

for paper_id, title, file_path in paper_rows:
    print(f"Processing Paper ID={paper_id} : {title}")

    # 1) Extract text + find abstract
    full_text = extract_text_from_pdf(file_path)
    abstract = get_first_page_text(file_path)

    # 2) Upsert doc into Chroma (metadata includes paper_id)
    #    (In a real scenario, you might chunk the text.)
    metadata = {"paper_id": paper_id, "title": title}
    doc = Document(page_content=full_text, metadata=metadata)
    vectorstore.add_documents([doc])
    # vectorstore.persist()

    # 3) Build the classification prompt
    prompt = CLASSIFICATION_PROMPT.format(
        title=title,
        abstract=abstract
    )

    # 4) Run the agent
    print("Asking the agent if the paper is about 'development'...")
    try:
        agent_response = agent.run(prompt)
        # Expecting "YES" or "NO" in the response.
        print("Agent response:", agent_response)

        # Convert to boolean
        is_dev = False
        if "YES" in agent_response.upper():
            is_dev = True

        # 5) Upsert or update in paper_assessments
        #    Check if a row exists already
        cursor.execute("SELECT paper_id FROM paper_assessments WHERE paper_id=?", (paper_id,))
        existing = cursor.fetchone()

        if not existing:
            # Insert new
            cursor.execute("""
                INSERT INTO paper_assessments (
                    paper_id,
                    is_development,
                    assessment_date
                )
                VALUES (?, ?, ?)
            """, (paper_id, is_dev, datetime.now()))
        else:
            # Update existing
            cursor.execute("""
                UPDATE paper_assessments
                SET is_development = ?,
                    assessment_date = ?
                WHERE paper_id = ?
            """, (is_dev, datetime.now(), paper_id))

        # Mark the paper as processed if desired
        cursor.execute("UPDATE papers SET processed=1 WHERE id=?", (paper_id,))
        conn.commit()

    except Exception as e:
        print(f"Error classifying paper {paper_id}: {e}")

# Close DB
cursor.close()
conn.close()